This is a separate file for just testing plotting and preprocessing code without worrying about models.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os 
import sys 
sys.path.append('..')

import matplotlib.pyplot as plt 
import numpy as np

import hippocampalseq as hse
import hippocampalseq.preprocessing as hsep
import hippocampalseq.analysis as hsea
import hippocampalseq.models as hsem 
import hippocampalseq.utils as hseu
import hippocampalseq.plotting as hsepl

In [ ]:
theta_time_window_ms = 60#10#250
theta_time_window_s  = theta_time_window_ms / 1000
theta_time_window_advance_ms = 60#5#250
theta_time_window_advance_s  = theta_time_window_advance_ms / 1000

ripple_time_window_ms = 3.0#5.0
ripple_time_window_s  = ripple_time_window_ms / 1000


bin_size = 2
data_path = os.path.realpath("../data")
rat_name = "Janni"
session = 1
track_type = "Linear"
results_path = f"../results/{rat_name}/{track_type}{session}"

In [ ]:
(
    raw_data,
    place_field_data,
) = hse.load_raw_data(
    data_path,
    rat_name,
    session,
    track_type=track_type,
    environment_size=environment_size,
    bin_size_cm=bin_size,
    placefield_kwargs = {
        'place_field_posterior': False,
        'velocity_cutoff': 10.0,
        "flatten_linear": False
    },

)

theta_data = hse.process_theta(
    raw_data,
    place_field_data, 
    velocity_cutoff=10.0,
    theta_kwargs = {
        'time_window_ms': theta_time_window_ms,
        'time_window_advance_ms': theta_time_window_advance_ms 
    }
)

In [ ]:
total_duration = raw_data.raw_position.time_support.tot_length('s')

firing_rate_per_phase,phase_centers = hsea.calculate_phase_locking(
    theta_data.spikes_with_phase,
    total_duration,
    5.0
)
modality_results = hsea.classify_theta_modality(
    firing_rate_per_phase,
    theta_data.spikes_with_phase,
)
population_stats = hsea.calculate_population_firing_rates(
    firing_rate_per_phase,
    modality_results,
    raw_data.excitatory_neurons
)
excitatory_set      = set(raw_data.excitatory_neurons)
true_excit_in_phase = sorted(set(theta_data.spikes_with_phase.keys()) & excitatory_set)

(
    pooled_cells,
    unimodal_cells,
    bimodal_cells 
) = hsea.classify_place_cell_modality(
    place_field_data.place_fields,
    place_field_data.place_cell_ids,
    place_field_data.position_hist,
    theta_data.spikes_with_phase,
    modality_results,
    true_excit_in_phase,
    5.0
)

In [ ]:
hsepl.plot_trajectory_with_velocity(
    raw_data.raw_position[['x', 'y']].values[:1000],
    raw_data.raw_position['Velocity'].values[:1000],
    raw_data.environment_size
)

In [ ]:
_=hsepl.spike_raster_plot(
    raw_data.raw_spikes,
    plot_end_time=32700
)

In [ ]:
_=hsepl.plot_lfp_data(
    raw_data.lfp_data
)

In [ ]:
hsepl.plot_session_stitching(
    raw_data.running_position,
    raw_data.running_spikes
)

In [ ]:
_=hsepl.plot_phase_locked(
    firing_rate_per_phase,
    phase_centers
)

In [ ]:
_=hsepl.plot_modality_classification(
    firing_rate_per_phase,
    modality_results,
    population_stats,
    phase_centers
)

In [ ]:
_=hsepl.plot_modality_all_cells(
    firing_rate_per_phase,
    modality_results,
    phase_centers
)

In [ ]:
_=hsepl.plot_place_field_comparison(
    unimodal_cells,
    bimodal_cells,
    f"{rat_name} {track_type}{session}"
)

In [ ]:
_=hsepl.plot_unimodal_bimodal_summary(
    modality_results,
    population_stats,
    phase_centers,
    unimodal_cells,
    bimodal_cells,
    raw_data.excitatory_neurons
)

In [ ]:
_=hsepl.plot_modality_overlay(population_stats, phase_centers)

In [ ]:
_=hsepl.plot_modality_pie(modality_results)

In [ ]:
_=hsepl.plot_individual_cell_histograms(
    firing_rate_per_phase,
    modality_results,
    phase_centers
)

In [ ]:
_=hsepl.plot_theta_phase_assignment(
    theta_data.spikes_with_phase,
    theta_data.lfp_data,
    theta_data.trough_indices,
    raw_data.excitatory_neurons,
    time_window=2
)

In [ ]:
_=hsepl.plot_cell_phase_polar(theta_data.spikes_with_phase)

In [ ]:
_=hsepl.plot_theta_lfp_segment(
    theta_data.lfp_data,
    theta_data.trough_times,
    theta_data.trough_indices,
    2.0
)

In [ ]:
_=hsepl.plot_theta_cycle_dist(
    theta_data.lfp_data
)

In [ ]:
print(np.max(place_field_data.place_fields[:4], axis=(1,2)))
hsepl.plot_open_placefields(
    place_field_data.place_fields,
    file_path=results_path,
    show_titles=False
)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10,10))
hsepl.plot_trajectories(theta_data.true_trajectory)

In [ ]:
_=hsepl.plot_spikemat_position_aligned(
    raw_data.running_spike_info,
    raw_data.raw_position,
    place_field_data.place_cell_ids,
    n_cells=5,
    cell_selection=[77]#'random'
)